In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Create directory
!mkdir -p /content/bobiac_data_cellpose
# Download the data
!wget https://github.com/bobiac/bobiac-book/releases/download/data-bobiac-2026/04_05_06_07_seg_and_spot.zip -O /content/bobiac_data_cellpose/04_05_06_07_seg_and_spot.zip
# Unzip the data, remove zip file and macOS metadata files (if any)
!cd /content/bobiac_data_cellpose && unzip 04_05_06_07_seg_and_spot.zip && rm -f 04_05_06_07_seg_and_spot.zip && rm -rf __MACOSX

In [ ]:
!pip install cellpose
!pip install matplotlib
!pip install tqdm

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from cellpose import core, io, models, plot
from cellpose.models import MODEL_DIR
from tqdm import tqdm

In [ ]:
io.logger_setup()  # to get printing of progress

use_gpu = core.use_gpu()
print("GPU available:", use_gpu)

In [ ]:
image_path = "content/bobiac_data_cellpose/04_segmentation_cellpose/cell_cellpose.tif"
image = io.imread(image_path)

In [ ]:
print(image.shape)

In [ ]:
ch = 1
plt.imshow(image[ch], cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
from cellpose.utils import download_url_to_file

model_name = "cpsam_v2"  # or "cpdino" / "cpdino-vitb"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / model_name
if not model_path.exists():
    url = f"https://huggingface.co/mouseland/cellpose-sam/resolve/main/{model_name}"
    download_url_to_file(url, str(model_path))

In [ ]:
model_path = str(MODEL_DIR / "cpsam_v2")  # or "cpdino" / "cpdino-vitb" or "cpsam"
model = models.CellposeModel(pretrained_model=model_path, gpu=use_gpu)

In [ ]:
cp_image = image[[0, 1]]
masks, flows, styles = model.eval(cp_image)

In [ ]:
fig = plt.figure(figsize=(12, 5))
plot.show_segmentation(fig, cp_image, masks, flows[0])
plt.tight_layout()
# Optional if you want to also save the figure
# plt.savefig(f"path/to/output/{Path(image_path).stem}_cp_output.png")
plt.show()

In [ ]:
output_path = f"path/to/output/{Path(image_path).stem}_labels.tif"
io.imsave(output_path, masks)  # or tifffile.imwrite(output_path, masks)

In [ ]:
# Path to the folder containing the images to segment
folder_path = Path("content/bobiac_data_cellpose/04_segmentation_cellpose")

# Create a subfolder to save the cell segmentation results
labels_folder = folder_path / "cell_labels"
labels_folder.mkdir(exist_ok=True)

# Get the sorted list of all .tif images in the folder
images_path = sorted(folder_path.glob("*.tif"))

# Initialize the model once before the loop
model_path = str(MODEL_DIR / "cpsam_v2")  # or "cpdino" / "cpdino-vitb" or "cpsam"
model = models.CellposeModel(pretrained_model=model_path, gpu=use_gpu)

# Run Cellpose on each image one by one
# NOTE: tqdm is used to show a progress bar, but you can remove it if you don't want it
for image_path in tqdm(images_path, desc="Processing images"):
    # Load the image
    image = io.imread(image_path)
    # Run Cellpose on the image
    masks, flows, styles = model.eval(image)
    # Save the segmentation results as a TIFF file
    output_path = labels_folder / f"{image_path.stem}_labels.tif"
    io.imsave(output_path, masks)  # or tifffile.imwrite(output_path, masks)

In [ ]:
# Download the data
!wget https://github.com/bobiac/bobiac-book/releases/download/data-bobiac-2026/04_segmentation_cellpose_3d.zip -O /content/bobiac_data_cellpose/04_segmentation_cellpose_3d.zip
# Unzip the data, remove zip file and macOS metadata files (if any)
!cd /content/bobiac_data_cellpose && unzip 04_segmentation_cellpose_3d.zip && rm -f 04_segmentation_cellpose_3d.zip && rm -rf __MACOSX

In [ ]:
image_path = (
    "/content/bobiac_data_cellpose/04_segmentation_cellpose/cell_cellpose_3d_crop.tif"
)
image = io.imread(image_path)  # or image = tifffile.imread(image_path)

print(image.shape)

In [ ]:
scale_x = 0.202
scale_y = 0.202
scale_z = 0.5
anisotropy = scale_z / scale_x
print("Anisotropy:", anisotropy)

In [ ]:
# Create the max intensity projection image
max_p = np.max(image, axis=0)

# Visualize the max intensity projection image
plt.imshow(max_p, cmap="gray")
plt.axis("off")
plt.tight_layout()
plt.show()

# We can also visualise the reslice of the image acros the XZ dimension
plt.imshow(image[:, 110, :], cmap="gray")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
model_path = str(MODEL_DIR / "cpsam_v2")  # or "cpdino" / "cpdino-vitb" or "cpsam"
model = models.CellposeModel(pretrained_model=model_path, gpu=use_gpu)

In [ ]:
masks, flows, styles = model.eval(
    image, do_3D=True, z_axis=0, anisotropy=anisotropy, flow3D_smooth=1
)

In [ ]:
z = 18  # pick a slice
fig = plt.figure(figsize=(12, 5))
plot.show_segmentation(fig, image[z], masks[z], flows[0][z])
plt.tight_layout()
plt.show()

In [ ]:
output_path = f"path/to/output/{Path(image_path).stem}_3d_labels.tif"
io.imsave(output_path, masks)  # or tifffile.imwrite(output_path, masks)

In [ ]:
# Path to the folder containing the images to segment
folder_path = Path("content/bobiac_data_cellpose/04_segmentation_cellpose")

# Create a subfolder to save the cell segmentation results
labels_folder = folder_path / "cell_labels"
labels_folder.mkdir(exist_ok=True)

# Get the sorted list of all .tif images in the folder
images_path = sorted(folder_path.glob("*.tif"))

# Increase batch_size to reduce GPU passes per image (uses more GPU memory)
batch_size = 8  # each pass sends batch_size tiles of 256×256 to the GPU
for image_path in tqdm(images_path, desc="Processing images"):
    image = io.imread(image_path)
    masks, flows, styles = model.eval(image, batch_size=batch_size)
    output_path = labels_folder / f"{image_path.stem}_labels.tif"
    io.imsave(output_path, masks)